# Projeto 1 — Demand Drivers & Commercial Forecast

## Electronics e Personal Care

### Objetivo
Investigar se variáveis comerciais ajudam a explicar a demanda e construir um modelo de previsão condicionado a fatores como:

- preço;
- desconto;
- promoção;
- estoque;
- categoria;
- calendário.

Nesta fase, restringimos a análise a **Electronics** e **Personal Care**, categorias nas quais o forecast univariado apresentou uma dinâmica futura visualmente mais coerente com a oscilação histórica.

> **Importante:** associação não significa causalidade. O objetivo é avaliar capacidade explicativa e preditiva, não provar que uma variável comercial causou a variação de vendas.


## Por que existe esta versão corrigida?

Esta versão surgiu como resultado direto do próprio processo de análise.

Na etapa anterior de forecasting, os gráficos de **Electronics** e **Personal Care** apresentaram oscilações semanais relevantes e um comportamento futuro que justificava uma investigação mais detalhada. Em vez de assumir imediatamente que essas variações eram explicadas por fatores como promoção, desconto, preço ou estoque, foi criado um notebook específico para investigar os números e testar essa hipótese.

A investigação mostrou que uma parte importante da volatilidade dos totais semanais estava associada à **quantidade de registros disponíveis em cada semana**. Em outras palavras, semanas com mais observações tendiam naturalmente a apresentar maiores volumes agregados de `units_sold`, o que poderia levar a uma interpretação incorreta dos picos e vales como mudanças reais de demanda.

Além disso, o primeiro cenário futuro havia criado combinações de **loja × dia** em uma densidade muito superior à observada historicamente. Ao somar essas previsões, o resultado ficou artificialmente inflado em relação à escala original da base.

Por esse motivo, esta versão foi construída para incorporar o resultado dessa investigação e corrigir a metodologia de projeção. O forecast futuro passa a preservar uma **densidade de observações semelhante à histórica**, permitindo comparar cenários comerciais em uma escala mais coerente com os dados disponíveis.

Portanto, esta versão representa uma evolução do projeto baseada em:

1. observação dos resultados iniciais;
2. formulação de uma hipótese sobre a volatilidade;
3. investigação dos números em um notebook complementar;
4. identificação de uma limitação metodológica;
5. revisão do modelo de projeção.

> Esse processo é parte importante do projeto: o objetivo não foi apenas produzir uma previsão, mas **questionar o resultado, testar sua consistência e revisar a abordagem quando os dados indicaram essa necessidade**.


## 1. Bibliotecas e leitura da base tratada


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ARQUIVO = "retail_sales_cleaned_project1.csv"

df = pd.read_csv(
    ARQUIVO,
    parse_dates=["date"]
)

categorias = ["Electronics", "Personal Care"]

base = (
    df[df["category"].isin(categorias)]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Linhas selecionadas: {len(base):,}")
print(f"Período: {base['date'].min().date()} a {base['date'].max().date()}")
display(base.head())


## 2. Diagnóstico da volatilidade semanal

Antes de relacionar picos de vendas com promoção, preço ou desconto, precisamos verificar se o total semanal pode estar sendo influenciado pela própria quantidade de observações existentes em cada semana.

Se uma semana possui mais linhas na base, naturalmente pode apresentar mais unidades vendidas quando usamos `sum(units_sold)`.

Por isso, calculamos:

- unidades totais;
- quantidade de registros;
- unidades médias por observação;
- preço médio;
- desconto médio;
- participação de promoções;
- estoque médio.


In [ ]:
weekly_diag = (
    base.set_index("date")
        .groupby("category")
        .resample("W-SUN")
        .agg(
            units_sold=("units_sold", "sum"),
            records=("units_sold", "size"),
            avg_units_per_record=("units_sold", "mean"),
            avg_price=("price", "mean"),
            avg_discount=("discount_percent", "mean"),
            avg_inventory=("inventory_level", "mean"),
            promotion_rate=(
                "promotion_active",
                lambda x: (x == "Yes").mean() * 100
            )
        )
        .reset_index()
)

display(weekly_diag.head())


### Total vendido × quantidade de registros

Este teste é importante para interpretar corretamente o gráfico semanal anterior.

Se a correlação entre `units_sold` e `records` for muito elevada, parte relevante dos picos e vales pode decorrer da quantidade de observações disponíveis em cada semana, e não necessariamente de uma mudança real de comportamento comercial.


In [ ]:
for categoria in categorias:
    temp = weekly_diag[weekly_diag["category"] == categoria]

    corr = temp[["units_sold", "records"]].corr().iloc[0, 1]

    print(
        f"{categoria}: correlação entre total semanal vendido "
        f"e quantidade de registros = {corr:.2f}"
    )

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["records"],
        temp["units_sold"],
        alpha=0.75
    )
    plt.title(f"Quantidade de registros x Total vendido — {categoria}")
    plt.xlabel("Registros na semana")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


### Decisão metodológica

Como o total semanal pode ser fortemente afetado pelo número de registros existentes na base, o modelo explicativo será construído no **nível de observação**.

Assim:

- variável-alvo: `units_sold`;
- cada linha continua representando uma observação comercial;
- os fatores comerciais são usados diretamente para prever a quantidade vendida;
- posteriormente, as previsões individuais são agregadas por semana para comparação com os valores observados.

Isso evita atribuir ao preço, desconto ou promoção uma volatilidade que pode ser causada apenas pela quantidade de linhas da amostra.


## 3. Investigação dos fatores comerciais


In [ ]:
base["promotion_flag"] = (
    base["promotion_active"]
    .eq("Yes")
    .astype(int)
)

drivers = [
    "units_sold",
    "price",
    "discount_percent",
    "inventory_level",
    "promotion_flag"
]

for categoria in categorias:
    print(f"\n### {categoria}")
    display(
        base.loc[base["category"] == categoria, drivers]
            .corr()
            .round(3)
    )


### Promoção × unidades vendidas

Comparamos a média de vendas entre observações com e sem promoção.

A diferença observada é uma associação descritiva e não uma estimativa causal do efeito da promoção.


In [ ]:
promotion_summary = (
    base.groupby(["category", "promotion_active"])
        .agg(
            observations=("units_sold", "size"),
            avg_units=("units_sold", "mean"),
            median_units=("units_sold", "median"),
            avg_discount=("discount_percent", "mean"),
            avg_price=("price", "mean")
        )
        .reset_index()
)

display(promotion_summary)


### Desconto × unidades vendidas


In [ ]:
discount_summary = (
    base.groupby(["category", "discount_percent"])
        .agg(
            observations=("units_sold", "size"),
            avg_units=("units_sold", "mean"),
            avg_price=("price", "mean")
        )
        .reset_index()
)

display(discount_summary)


In [ ]:
for categoria in categorias:
    temp = base[base["category"] == categoria]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["discount_percent"],
        temp["units_sold"],
        alpha=0.55
    )
    plt.title(f"Desconto x Unidades vendidas — {categoria}")
    plt.xlabel("Desconto (%)")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


### Preço × unidades vendidas


In [ ]:
for categoria in categorias:
    temp = base[base["category"] == categoria]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["price"],
        temp["units_sold"],
        alpha=0.55
    )
    plt.title(f"Preço x Unidades vendidas — {categoria}")
    plt.xlabel("Preço")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


### Estoque × unidades vendidas

O estoque disponível pode limitar a quantidade vendida. Ainda assim, uma correlação simples não é suficiente para concluir que estoque maior gera vendas maiores.


In [ ]:
for categoria in categorias:
    temp = base[base["category"] == categoria]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["inventory_level"],
        temp["units_sold"],
        alpha=0.55
    )
    plt.title(f"Estoque x Unidades vendidas — {categoria}")
    plt.xlabel("Nível de estoque")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


## 4. Picos e vales semanais

Agora observamos as semanas de maior e menor volume e os respectivos indicadores comerciais.

O objetivo é verificar se os extremos apresentam padrões semelhantes de promoção, desconto, preço ou estoque.


In [ ]:
for categoria in categorias:

    temp = (
        weekly_diag[weekly_diag["category"] == categoria]
        .sort_values("units_sold", ascending=False)
    )

    print(f"\n{categoria} — 5 maiores semanas")
    display(temp.head(5))

    print(f"{categoria} — 5 menores semanas")
    display(temp.tail(5))


## 5. Preparação das features para previsão

O modelo utilizará apenas informações que poderiam ser conhecidas ou planejadas comercialmente:

### Comerciais
- `price`
- `discount_percent`
- `inventory_level`
- `promotion_active`

### Contexto
- `category`
- `store_id`
- dia da semana
- mês
- fim de semana

`product_id` não será utilizado nesta fase porque existem muitos produtos com poucas observações, o que aumentaria muito a dimensionalidade e o risco de overfitting.


In [ ]:
model_data = base.copy()

model_data["month_num"] = model_data["date"].dt.month
model_data["day_name"] = model_data["date"].dt.day_name()
model_data["is_weekend_model"] = (
    model_data["date"].dt.dayofweek >= 5
).astype(int)

# Representação cíclica do mês
model_data["month_sin"] = np.sin(
    2 * np.pi * model_data["month_num"] / 12
)
model_data["month_cos"] = np.cos(
    2 * np.pi * model_data["month_num"] / 12
)

features = [
    "price",
    "discount_percent",
    "inventory_level",
    "promotion_active",
    "category",
    "store_id",
    "day_name",
    "is_weekend_model",
    "month_sin",
    "month_cos"
]

target = "units_sold"

X = model_data[features].copy()
y = model_data[target].copy()


## 6. Separação temporal treino/teste

Não utilizaremos divisão aleatória.

O conjunto de teste será formado pelas observações das últimas 8 semanas do histórico, preservando a ordem temporal e simulando uma previsão real de período futuro.


In [ ]:
cutoff_date = model_data["date"].max() - pd.Timedelta(weeks=8)

train_mask = model_data["date"] <= cutoff_date
test_mask = model_data["date"] > cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

meta_test = model_data.loc[
    test_mask,
    ["date", "category", "store_id"]
].copy()

print(f"Data de corte: {cutoff_date.date()}")
print(f"Treino: {len(X_train):,} observações")
print(f"Teste: {len(X_test):,} observações")


## 7. Modelos comerciais

Serão comparados dois modelos:

### Ridge Regression
Modelo linear regularizado, útil como referência interpretável.

### Random Forest
Modelo não linear capaz de capturar interações entre desconto, promoção, preço, estoque, loja e calendário.

O objetivo não é escolher o modelo mais complexo, mas verificar se os fatores comerciais oferecem capacidade preditiva relevante.


In [ ]:
numeric_features = [
    "price",
    "discount_percent",
    "inventory_level",
    "is_weekend_model",
    "month_sin",
    "month_cos"
]

categorical_features = [
    "promotion_active",
    "category",
    "store_id",
    "day_name"
]

preprocessor_linear = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

ridge_model = Pipeline([
    ("preprocessor", preprocessor_linear),
    ("model", Ridge(alpha=1.0))
])

rf_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        )
    )
])


In [ ]:
ridge_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

pred_ridge = ridge_model.predict(X_test)
pred_rf = rf_model.predict(X_test)


## 8. Métricas

Além do MAE, RMSE, MAPE e Bias, as métricas serão calculadas também por categoria.


In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        )
        * 100
    )


def bias(y_true, y_pred):
    return np.mean(
        np.asarray(y_pred)
        - np.asarray(y_true)
    )


def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAPE": mape(y_true, y_pred),
        "Bias": bias(y_true, y_pred)
    }


In [ ]:
overall_results = pd.DataFrame([
    {
        "Model": "Ridge",
        **metrics(y_test, pred_ridge)
    },
    {
        "Model": "Random Forest",
        **metrics(y_test, pred_rf)
    }
])

display(overall_results)


In [ ]:
predictions = meta_test.copy()
predictions["actual"] = y_test.values
predictions["Ridge"] = pred_ridge
predictions["Random Forest"] = pred_rf

category_results = []

for categoria in categorias:

    temp = predictions[
        predictions["category"] == categoria
    ]

    for modelo in ["Ridge", "Random Forest"]:

        category_results.append({
            "Category": categoria,
            "Model": modelo,
            **metrics(
                temp["actual"],
                temp[modelo]
            )
        })

category_results = pd.DataFrame(category_results)

display(
    category_results
    .sort_values(["Category", "MAPE"])
)


## 9. Actual vs Predicted — agregado por semana

As previsões são feitas em nível de observação e depois agregadas semanalmente.

Esse gráfico permite comparar a forma da demanda observada com a demanda prevista sem utilizar diretamente o número de registros como variável explicativa.


In [ ]:
weekly_predictions = (
    predictions
    .groupby(["date", "category"], as_index=False)
    .agg(
        actual=("actual", "sum"),
        Ridge=("Ridge", "sum"),
        Random_Forest=("Random Forest", "sum")
    )
)

# A agregação acima ainda está por data.
# Reagregamos para semana.
weekly_predictions = (
    weekly_predictions
    .set_index("date")
    .groupby("category")
    .resample("W-SUN")
    .agg(
        actual=("actual", "sum"),
        Ridge=("Ridge", "sum"),
        Random_Forest=("Random_Forest", "sum")
    )
    .reset_index()
)

display(weekly_predictions.head())


In [ ]:
for categoria in categorias:

    temp = weekly_predictions[
        weekly_predictions["category"] == categoria
    ]

    plt.figure(figsize=(10, 4))

    plt.plot(
        temp["date"],
        temp["actual"],
        marker="o",
        label="Actual"
    )

    plt.plot(
        temp["date"],
        temp["Ridge"],
        marker="o",
        linestyle="--",
        label="Ridge"
    )

    plt.plot(
        temp["date"],
        temp["Random_Forest"],
        marker="o",
        linestyle="--",
        label="Random Forest"
    )

    plt.title(
        f"Actual vs Commercial Models — {categoria}"
    )
    plt.xlabel("Semana")
    plt.ylabel("Unidades vendidas")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Importância das variáveis — Random Forest

Usamos **Permutation Importance** diretamente sobre o conjunto de teste.

A medida responde:

> Quanto o desempenho do modelo piora quando embaralhamos uma determinada variável?

Isso ajuda a identificar quais informações tiveram maior utilidade preditiva no período de teste.

A importância continua sendo **preditiva**, não causal.


In [ ]:
perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance": perm.importances_mean
    })
    .sort_values("importance", ascending=True)
)

display(
    importance.sort_values(
        "importance",
        ascending=False
    )
)


In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.barh(
    importance["feature"],
    importance["importance"]
)

plt.title(
    "Importância preditiva das variáveis — Random Forest"
)
plt.xlabel("Queda de desempenho após permutação")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 11. Investigação específica por categoria

Além da importância global, verificamos novamente os principais indicadores descritivos separadamente para Electronics e Personal Care.

Isso é importante porque a relação entre desconto e demanda pode ser diferente entre categorias.


In [ ]:
for categoria in categorias:

    temp = base[
        base["category"] == categoria
    ]

    print(f"\n{categoria}")

    resumo = pd.DataFrame({
        "Métrica": [
            "Unidades médias",
            "Preço médio",
            "Desconto médio",
            "% observações em promoção",
            "Estoque médio"
        ],
        "Valor": [
            temp["units_sold"].mean(),
            temp["price"].mean(),
            temp["discount_percent"].mean(),
            temp["promotion_flag"].mean() * 100,
            temp["inventory_level"].mean()
        ]
    })

    display(resumo)


# 12. Forecast condicionado a cenários comerciais

Há uma diferença importante entre um forecast univariado e este modelo:

### Holt-Winters
Pode projetar o futuro apenas a partir do comportamento passado.

### Modelo com fatores comerciais
Para prever o futuro, precisamos informar quais serão as condições comerciais futuras:

- preço;
- desconto;
- promoção;
- estoque;
- loja;
- calendário.

Por isso, esta etapa gera **previsões condicionais**, e não uma única previsão inevitável.

A seguir criamos uma estrutura de cenário que pode ser modificada manualmente.


In [ ]:
# Escolhemos o modelo comercial com menor MAPE geral
best_model_name = (
    overall_results
    .sort_values("MAPE")
    .iloc[0]["Model"]
)

best_model = (
    ridge_model
    if best_model_name == "Ridge"
    else rf_model
)

print("Modelo selecionado:", best_model_name)


## 13. Correção do cenário futuro: preservar a densidade histórica de observações

Na versão anterior, o cenário futuro criou uma linha para **cada loja × cada dia**.  
Isso produziu muito mais observações do que existem historicamente e, consequentemente, inflou o total previsto de unidades.

A correção desta etapa segue uma regra simples:

> **Cada semana futura deve possuir uma quantidade de observações semelhante à quantidade normalmente observada no histórico.**

Para isso:

1. calculamos o número de registros por semana e categoria;
2. usamos a **mediana das últimas 8 semanas** como referência;
3. criamos somente essa quantidade de observações em cada semana futura;
4. preservamos a distribuição recente de lojas e dias da semana por amostragem;
5. alteramos apenas as premissas comerciais do cenário.

Assim, o volume previsto passa a ser comparável com a escala histórica.


In [ ]:
FORECAST_WEEKS = 8
RECENT_WEEKS = 8
RANDOM_STATE = 42

recent_cutoff = (
    base["date"].max()
    - pd.Timedelta(weeks=RECENT_WEEKS)
)

recent = (
    base[base["date"] > recent_cutoff]
    .copy()
)

# Quantidade histórica de observações por semana e categoria
recent_weekly_records = (
    recent.set_index("date")
          .groupby("category")
          .resample("W-SUN")
          .size()
          .rename("records")
          .reset_index()
)

records_reference = (
    recent_weekly_records
    .groupby("category")["records"]
    .agg(
        median_records="median",
        mean_records="mean",
        min_records="min",
        max_records="max"
    )
)

display(records_reference)


### Interpretação

A mediana é utilizada porque é menos sensível a semanas excepcionalmente cheias ou vazias.

Essa quantidade não representa necessariamente o número real de transações de uma rede varejista.  
Ela representa a **densidade observacional da base disponível**, que precisa ser preservada para que a soma das previsões permaneça comparável ao histórico.


In [ ]:
# Referências comerciais recentes por categoria
commercial_reference = (
    recent.groupby("category")
          .agg(
              median_price=("price", "median"),
              median_discount=("discount_percent", "median"),
              median_inventory=("inventory_level", "median"),
              promotion_rate=(
                  "promotion_active",
                  lambda x: (x == "Yes").mean()
              )
          )
)

display(commercial_reference)


## 14. Construção de cenários comerciais

Criaremos três cenários:

- **Base**: mantém preço, desconto, estoque e promoção próximos ao padrão recente;
- **Desconto +5 p.p.**: aumenta o desconto em 5 pontos percentuais;
- **Desconto +10 p.p.**: aumenta o desconto em 10 pontos percentuais.

Neste momento mantemos as demais premissas constantes para isolar o comportamento estimado associado ao desconto.

> Os cenários são simulações condicionais. Eles não representam causalidade nem garantem que a demanda responderá dessa forma no mundo real.


In [ ]:
SCENARIOS = {
    "Base": 0,
    "Discount +5pp": 5,
    "Discount +10pp": 10
}

last_date = base["date"].max()

# Domingos que encerram as 8 semanas futuras
first_future_sunday = (
    last_date
    + pd.offsets.Week(weekday=6)
)

future_week_ends = pd.date_range(
    start=first_future_sunday,
    periods=FORECAST_WEEKS,
    freq="W-SUN"
)

future_week_ends


In [ ]:
def build_future_scenario(
    recent_data,
    records_reference,
    commercial_reference,
    scenario_name,
    discount_increment,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    rows = []

    for categoria in categorias:

        cat_recent = (
            recent_data[
                recent_data["category"] == categoria
            ]
            .copy()
        )

        # Número de observações a preservar por semana
        n_records = int(
            round(
                records_reference.loc[
                    categoria,
                    "median_records"
                ]
            )
        )

        ref = commercial_reference.loc[categoria]

        for week_number, week_end in enumerate(
            future_week_ends,
            start=1
        ):

            # Amostragem de observações recentes para preservar
            # aproximadamente o mix de lojas e dias da semana.
            sampled = cat_recent.sample(
                n=n_records,
                replace=True,
                random_state=(
                    random_state
                    + week_number
                    + (0 if categoria == categorias[0] else 1000)
                )
            ).copy()

            # Mantemos o dia da semana da observação amostrada,
            # mas deslocamos a data para a semana futura correspondente.
            sampled_day_num = sampled["date"].dt.dayofweek.to_numpy()

            week_start = (
                week_end
                - pd.Timedelta(days=6)
            )

            future_dates = [
                week_start
                + pd.Timedelta(days=int(day_num))
                for day_num in sampled_day_num
            ]

            # Promoção conforme proporção recente da categoria
            promo_flags = (
                rng.random(n_records)
                < ref["promotion_rate"]
            )

            scenario_discount = min(
                100,
                ref["median_discount"]
                + discount_increment
            )

            for i in range(n_records):
                rows.append({
                    "scenario": scenario_name,
                    "forecast_week": week_end,
                    "date": future_dates[i],
                    "category": categoria,
                    "store_id": sampled.iloc[i]["store_id"],
                    "price": ref["median_price"],
                    "discount_percent": scenario_discount,
                    "inventory_level": ref["median_inventory"],
                    "promotion_active": (
                        "Yes"
                        if promo_flags[i]
                        else "No"
                    )
                })

    future = pd.DataFrame(rows)

    future["month_num"] = future["date"].dt.month
    future["day_name"] = future["date"].dt.day_name()

    future["is_weekend_model"] = (
        future["date"].dt.dayofweek >= 5
    ).astype(int)

    future["month_sin"] = np.sin(
        2 * np.pi
        * future["month_num"]
        / 12
    )

    future["month_cos"] = np.cos(
        2 * np.pi
        * future["month_num"]
        / 12
    )

    return future


In [ ]:
scenario_frames = []

for scenario_name, discount_increment in SCENARIOS.items():

    temp = build_future_scenario(
        recent_data=recent,
        records_reference=records_reference,
        commercial_reference=commercial_reference,
        scenario_name=scenario_name,
        discount_increment=discount_increment,
        random_state=RANDOM_STATE
    )

    scenario_frames.append(temp)

future_scenarios = pd.concat(
    scenario_frames,
    ignore_index=True
)

print("Total de observações futuras:", len(future_scenarios))

display(
    future_scenarios.groupby(
        ["scenario", "category", "forecast_week"]
    ).size().rename("records").reset_index().head(20)
)


## 15. Previsão condicionada aos cenários

Aplicamos o modelo comercial vencedor às observações futuras simuladas.

Depois, somamos as previsões por semana e categoria.

Como o número de observações semanais foi preservado, os totais agora ficam na mesma ordem de grandeza do histórico.


In [ ]:
X_future = future_scenarios[features]

future_scenarios["forecast_units"] = np.maximum(
    best_model.predict(X_future),
    0
)

future_weekly = (
    future_scenarios
    .groupby(
        ["scenario", "category", "forecast_week"],
        as_index=False
    )
    .agg(
        forecast_units=("forecast_units", "sum"),
        records=("forecast_units", "size"),
        avg_forecast_per_record=("forecast_units", "mean"),
        assumed_price=("price", "mean"),
        assumed_discount=("discount_percent", "mean"),
        assumed_inventory=("inventory_level", "mean"),
        promotion_rate=(
            "promotion_active",
            lambda x: (x == "Yes").mean() * 100
        )
    )
)

display(future_weekly.head(20))


## 16. Validação de escala

Antes de interpretar os cenários, comparamos a escala prevista com o histórico recente.

O objetivo não é exigir valores idênticos, mas confirmar que a previsão futura não foi inflada simplesmente por uma quantidade artificialmente maior de registros.


In [ ]:
historical_scale = (
    weekly_diag[
        weekly_diag["category"].isin(categorias)
    ]
    .groupby("category")
    .agg(
        historical_weekly_median=("units_sold", "median"),
        historical_weekly_mean=("units_sold", "mean"),
        historical_weekly_min=("units_sold", "min"),
        historical_weekly_max=("units_sold", "max")
    )
)

base_scale = (
    future_weekly[
        future_weekly["scenario"] == "Base"
    ]
    .groupby("category")
    .agg(
        forecast_weekly_median=("forecast_units", "median"),
        forecast_weekly_mean=("forecast_units", "mean"),
        forecast_weekly_min=("forecast_units", "min"),
        forecast_weekly_max=("forecast_units", "max")
    )
)

scale_check = historical_scale.join(base_scale)

display(scale_check)


## 17. Histórico × cenário-base corrigido


In [ ]:
for categoria in categorias:

    hist = weekly_diag[
        weekly_diag["category"] == categoria
    ].copy()

    future = future_weekly[
        (future_weekly["category"] == categoria)
        & (future_weekly["scenario"] == "Base")
    ].copy()

    plt.figure(figsize=(11, 4))

    plt.plot(
        hist["date"],
        hist["units_sold"],
        label="Histórico"
    )

    plt.plot(
        future["forecast_week"],
        future["forecast_units"],
        marker="o",
        linestyle="--",
        label="Cenário-base corrigido"
    )

    plt.axvline(
        hist["date"].max(),
        linestyle=":",
        label="Início do forecast"
    )

    plt.title(
        f"Histórico x Forecast Comercial Corrigido — {categoria}"
    )

    plt.xlabel("Semana")
    plt.ylabel("Unidades vendidas")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 18. Comparação entre cenários de desconto

Agora podemos comparar os três cenários mantendo a mesma quantidade de observações e alterando somente a premissa de desconto.

A diferença entre as linhas representa a resposta estimada pelo modelo sob cada condição comercial.


In [ ]:
for categoria in categorias:

    temp = future_weekly[
        future_weekly["category"] == categoria
    ].copy()

    plt.figure(figsize=(10, 4))

    for scenario in SCENARIOS.keys():

        scenario_data = temp[
            temp["scenario"] == scenario
        ]

        plt.plot(
            scenario_data["forecast_week"],
            scenario_data["forecast_units"],
            marker="o",
            label=scenario
        )

    plt.title(
        f"Cenários de desconto — {categoria}"
    )

    plt.xlabel("Semana")
    plt.ylabel("Unidades previstas")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 19. Impacto acumulado dos cenários

Para facilitar a futura análise financeira, agregamos as 8 semanas de cada cenário.


In [ ]:
scenario_summary = (
    future_weekly
    .groupby(["category", "scenario"], as_index=False)
    .agg(
        forecast_units_8w=("forecast_units", "sum"),
        avg_weekly_units=("forecast_units", "mean"),
        assumed_discount=("assumed_discount", "mean"),
        assumed_price=("assumed_price", "mean")
    )
)

base_values = (
    scenario_summary[
        scenario_summary["scenario"] == "Base"
    ][
        ["category", "forecast_units_8w"]
    ]
    .rename(
        columns={
            "forecast_units_8w": "base_units_8w"
        }
    )
)

scenario_summary = scenario_summary.merge(
    base_values,
    on="category",
    how="left"
)

scenario_summary["unit_change_vs_base"] = (
    scenario_summary["forecast_units_8w"]
    - scenario_summary["base_units_8w"]
)

scenario_summary["unit_change_pct_vs_base"] = (
    scenario_summary["unit_change_vs_base"]
    / scenario_summary["base_units_8w"]
    * 100
)

display(scenario_summary)


# 20. Conclusão metodológica desta correção

A versão corrigida resolve o principal problema identificado na projeção anterior:

- o forecast não cria mais todas as combinações possíveis de loja e dia;
- cada semana futura conserva uma quantidade de observações semelhante ao histórico;
- a composição de lojas e dias é amostrada do período recente;
- as alterações de cenário ficam concentradas nas premissas comerciais.

Com isso, a previsão passa a responder uma pergunta mais adequada:

> **Mantendo uma estrutura de observações semelhante à histórica, como a demanda estimada varia sob diferentes condições comerciais?**

Ainda assim, esses resultados devem ser tratados como **simulações preditivas condicionais**, e não como estimativas causais do efeito de desconto.


## 21. Exportação

São exportados:

- resultados gerais dos modelos;
- resultados por categoria;
- importância das variáveis;
- forecast futuro detalhado;
- forecast semanal por cenário;
- resumo acumulado dos cenários.

Esses arquivos serão utilizados na etapa de planejamento financeiro.


In [ ]:
overall_results.to_csv(
    "commercial_model_results_overall.csv",
    index=False,
    encoding="utf-8-sig"
)

category_results.to_csv(
    "commercial_model_results_by_category.csv",
    index=False,
    encoding="utf-8-sig"
)

importance.sort_values(
    "importance",
    ascending=False
).to_csv(
    "commercial_driver_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

future_scenarios.to_csv(
    "commercial_future_scenarios_detailed.csv",
    index=False,
    encoding="utf-8-sig"
)

future_weekly.to_csv(
    "commercial_forecast_scenarios_8_weeks.csv",
    index=False,
    encoding="utf-8-sig"
)

scenario_summary.to_csv(
    "commercial_scenario_summary_8_weeks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos gerados:")
print("- commercial_model_results_overall.csv")
print("- commercial_model_results_by_category.csv")
print("- commercial_driver_importance.csv")
print("- commercial_future_scenarios_detailed.csv")
print("- commercial_forecast_scenarios_8_weeks.csv")
print("- commercial_scenario_summary_8_weeks.csv")
